# 130 · Fix the environment

**Level:** basic · **Topics:** environment, installation, applications-library, HDAWG, flux · **Type:** fix the bugs

**The code in this notebook is correct. Do not change it.** The Python environment is broken.

A user set up a PQSC, an SHFQC+ and an HDAWG, and gets this error on a standard experiment:

```
Signal 'q0/flux' sets a local oscillator, which is unsupported on this channel type.
Remove the 'local_oscillator' calibration from this signal.
```

Nobody set a local oscillator on the flux line.

## Your task

1. Create the environment of the user. It is described in [`environment/pyproject.toml`](environment/pyproject.toml).
   With [uv](https://docs.astral.sh/uv/getting-started/installation/):

   ```bash
   cd exercises/130_fix_the_environment/environment
   uv sync
   uv run jupyter lab ../exercise.ipynb
   ```

2. Run the notebook and reproduce the error.
3. Find what is wrong with the environment.
4. Fix `environment/pyproject.toml`, run `uv sync` again, restart the kernel and run the notebook again.

In the standard environment of this repository the notebook runs without error: this is how it should behave.

📖 [Installation](https://docs.zhinst.com/labone_q_user_manual/getting_started/installation.html) ·
[Release notes](https://docs.zhinst.com/labone_q_user_manual/release_notes/index.html)

In [ ]:
import numpy as np

from laboneq.dsl.quantum import QPU
from laboneq.simple import *
from laboneq_applications.experiments import amplitude_rabi
from laboneq_applications.qpu_types.tunable_transmon import (
    TunableTransmonOperations,
    TunableTransmonQubit,
)

## Device setup

Drive and readout on the SHFQC+, the flux line on the HDAWG, synchronised by a PQSC.

📖 [Device setup](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/00_device_setup/concepts/index.html)

In [ ]:
setup = DeviceSetup("exercise_setup")
setup.add_dataserver(host="localhost", port="8004")
setup.add_instruments(
    SHFQC(uid="shfqc", address="DEV12001", device_options="SHFQC/PLUS/QC2CH"),
    HDAWG(uid="hdawg", address="DEV8001", device_options="HDAWG8/MF/ME/SKW/PC"),
    PQSC(uid="pqsc", address="DEV10001", reference_clock_source="internal"),
)
setup.add_connections(
    "shfqc",
    create_connection(to_signal="q0/drive", ports="SGCHANNELS/0/OUTPUT"),
    create_connection(to_signal="q0/measure", ports="QACHANNELS/0/OUTPUT"),
    create_connection(to_signal="q0/acquire", ports="QACHANNELS/0/INPUT"),
)
setup.add_connections(
    "hdawg",
    create_connection(to_signal="q0/flux", ports="SIGOUTS/0"),
)
setup.add_connections(
    "pqsc",
    create_connection(to_instrument="shfqc", ports="ZSYNCS/0"),
    create_connection(to_instrument="hdawg", ports="ZSYNCS/1"),
)

## Qubit and calibration

The calibration is generated from the qubit parameters.

📖 [Calibrating signals](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/02_logical_signals/concepts/00_calibrating_signals.html) ·
[Quantum operations](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/04_quantum_processing_unit/tutorials/00_quantum_operations.html)

In [ ]:
[q0] = TunableTransmonQubit.from_device_setup(setup)
q0.update(
    drive_lo_frequency=4.0e9,
    resonance_frequency_ge=4.1e9,
    readout_lo_frequency=7.0e9,
    readout_resonator_frequency=7.08e9,
)

qpu = QPU(quantum_elements=[q0], quantum_operations=TunableTransmonOperations())
setup.set_calibration(q0.calibration())

## Session

📖 [Session](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/01_session/concepts/index.html)

In [ ]:
session = Session(setup)
session.connect(do_emulation=True)

## Amplitude Rabi

A standard experiment of the Applications Library.

📖 [Amplitude Rabi workflow](https://docs.zhinst.com/labone_q_user_manual/applications_library/how-to-guides/sources/01_superconducting_qubits/01_workflows/04_amplitude_rabi.html)

In [ ]:
options = amplitude_rabi.experiment_workflow.options()
options.count(16)
options.do_analysis(False)

rabi = amplitude_rabi.experiment_workflow(
    session, qpu, "q0", amplitudes=np.linspace(0, 1, 11), options=options
)
result = rabi.run()

In [ ]:
# check
print(result.output["q0/result"].data.shape)

**Expected result:** `(11,)`

## Questions

1. Which package versions are installed in the broken environment?
2. Why does `uv` accept this combination?
3. Why does the mismatch give an error about a local oscillator?
4. Somebody proposes to patch the calibration after it is generated, to remove the local oscillator. Is this a good fix?

<details>
<summary>Hint 1</summary>

```python
from importlib.metadata import version

print(version("laboneq"), version("laboneq-applications"))
```
</details>

<details>
<summary>Hint 2</summary>

Read `environment/pyproject.toml` to the end. Then look at the flux line: `q0.calibration()["q0/flux"]`.
</details>

The solution is in [solution.ipynb](solution.ipynb).